# 01 · Fine-tune E5 retriever cho Research Desk
Học câu hỏi → đoạn bằng chứng bằng **CachedMultipleNegativesRankingLoss**. Mặc định E5-base, một GPU, logical batch 32 / mini-batch 4. Đây là notebook để bạn thực thi, chưa kèm kết quả GPU đã chạy.

### 1. Cài thư viện
Bật **Internet** và **GPU** trong Kaggle Settings. Chạy notebook từ đầu trong session mới; không cần key hay source ZIP. Nếu cell báo **THIẾT LẬP ĐÃ THÀNH CÔNG / RESTART KERNEL**, restart kernel/Python một lần, rồi chạy lại từ cell đầu. Không cần tạo lại phiên GPU. Đây là bước bắt buộc để tránh dùng chung NumPy cũ trong RAM và bản vừa cài trên đĩa.

In [ ]:
"""Notebook bootstrap: verify a fresh process and stop before using stale native modules."""

import hashlib
import importlib.metadata
import json
import os
import subprocess
import sys
import uuid
from pathlib import Path

TRAINING_REQUIREMENTS = [
    "sentence-transformers==5.1.2",
    "transformers==4.57.1",
    "datasets==4.3.0",
    "accelerate==1.11.0",
    "huggingface-hub==0.36.0",
    "pyarrow==21.0.0",
    "numpy==2.2.6",
    "sentencepiece==0.2.1",
]
WATCHED_MODULES = {
    "numpy": "numpy",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "pandas": "pandas",
    "pyarrow": "pyarrow",
    "torch": "torch",
    "transformers": "transformers",
    "sentence_transformers": "sentence-transformers",
    "datasets": "datasets",
    "huggingface_hub": "huggingface-hub",
    "tokenizers": "tokenizers",
    "accelerate": "accelerate",
    "safetensors": "safetensors",
}
PREFLIGHT_CODE = r"""
import json, sys
import numpy as np
from numpy import strings
assert strings.add(["a"], ["b"]).tolist() == ["ab"]
from scipy.sparse import csr_matrix
from sklearn.metrics import pairwise_distances
import pandas, pyarrow, datasets, torch
from sentence_transformers import (
    SentenceTransformer, CrossEncoder,
    SentenceTransformerTrainer, SentenceTransformerTrainingArguments,
)
from sentence_transformers.losses import CachedMultipleNegativesRankingLoss
assert csr_matrix([[1.0]]).shape == (1, 1)
assert pairwise_distances([[0.0], [1.0]]).shape == (2, 2)
print(json.dumps({"imports_ok": True, "python": sys.version.split()[0],
                  "numpy": np.__version__, "numpy_file": np.__file__,
                  "torch": torch.__version__, "cuda_available": torch.cuda.is_available()}))
"""


def installed_versions():
    versions = {}
    for name in set(WATCHED_MODULES.values()) | {r.split("==")[0] for r in TRAINING_REQUIREMENTS}:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions


def loaded_versions():
    return {
        name: vars(sys.modules[name]).get("__version__")
        for name in WATCHED_MODULES
        if name in sys.modules and sys.modules[name] is not None
    }


def current_runtime_identity():
    # A restarted container/kernel may reuse the same PID. A token stored on sys
    # survives re-running the cell, but cannot survive a real Python restart.
    if not getattr(sys, "_researchdesk_runtime_id", None):
        sys._researchdesk_runtime_id = f"{os.getpid()}:{uuid.uuid4().hex}"
    return sys._researchdesk_runtime_id


def install_training_environment(marker_path, runtime_id=None):
    """Never treat a successful pip install as proof the current kernel is coherent."""
    runtime_id = runtime_id or current_runtime_identity()
    marker_path = Path(marker_path)
    before, loaded = installed_versions(), loaded_versions()
    needed = [r for r in TRAINING_REQUIREMENTS if before[r.split("==")[0]] != r.split("==")[1]]
    if needed:
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--quiet",
                "--prefer-binary",
                "--only-binary=numpy,pyarrow",
                "--upgrade-strategy",
                "only-if-needed",
                *TRAINING_REQUIREMENTS,
            ],
            check=True,
        )
    # A child process has no old NumPy C-extension objects from the notebook kernel.
    checked = subprocess.run(
        [sys.executable, "-c", PREFLIGHT_CODE],
        capture_output=True,
        text=True,
        timeout=180,
    )
    repaired = False
    diagnostics = (checked.stdout or "") + (checked.stderr or "")
    if checked.returncode and any(
        text in diagnostics
        for text in (
            "numpy.ufunc",
            "numpy.dtype size changed",
            "numpy._core",
            "numpy.core",
        )
    ):
        # Repair only NumPy's files; do not reinstall Kaggle's CUDA PyTorch.
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--quiet",
                "--force-reinstall",
                "--no-cache-dir",
                "--no-deps",
                "--only-binary=:all:",
                "numpy==2.2.6",
            ],
            check=True,
        )
        repaired = True
        checked = subprocess.run(
            [sys.executable, "-c", PREFLIGHT_CODE],
            capture_output=True,
            text=True,
            timeout=180,
        )
    if checked.returncode:
        print((checked.stdout or "")[-4000:])
        print((checked.stderr or "")[-12000:])
        raise RuntimeError(
            "Fresh-process dependency check failed. Training has NOT started. "
            "Keep the complete output above for diagnosis."
        )
    after = installed_versions()
    fingerprint = hashlib.sha256(json.dumps(after, sort_keys=True).encode()).hexdigest()
    previous = json.loads(marker_path.read_text(encoding="utf-8")) if marker_path.exists() else {}
    changed = before != after
    mismatch = any(
        version is not None and str(version) != str(after.get(WATCHED_MODULES[name]))
        for name, version in loaded.items()
    )
    pending_same_kernel = previous.get("restart_required_for") == runtime_id
    unknown_loaded_kernel = bool(loaded) and (
        previous.get("fingerprint") != fingerprint
        or previous.get("runtime_identity_format") != "memory-token-v1"
    )
    restart = (
        pending_same_kernel
        or bool(loaded)
        and (changed or repaired or mismatch or unknown_loaded_kernel)
    )
    status = {
        "ready": not restart,
        "fingerprint": fingerprint,
        "versions": after,
        "runtime_id": runtime_id,
        "runtime_identity_format": "memory-token-v1",
        "restart_required_for": runtime_id if restart else None,
        "loaded_modules_before_install": sorted(loaded),
        "fresh_process_output": checked.stdout.strip(),
        "numpy_repaired": repaired,
    }
    marker_path.parent.mkdir(parents=True, exist_ok=True)
    marker_path.write_text(json.dumps(status, indent=2), encoding="utf-8")
    print(checked.stdout.strip())
    if restart:
        print(
            "THIẾT LẬP ĐÃ THÀNH CÔNG — cần RESTART KERNEL/PYTHON một lần.\n"
            "Đây là bước hoàn tất cài đặt, không phải lỗi huấn luyện.\n"
            "Sau restart, chạy lại từ cell đầu. Các cell phía sau sẽ tạm bỏ qua "
            "để tránh dùng thư viện cũ trong RAM."
        )
        return status
    print("Environment ready: fresh-process imports passed and no kernel restart is pending.")
    return status


import os, sys, subprocess, time
_RESEARCHDESK_ENV_READY = False
# This timer includes dependency installation and data/model preparation.
# If GPU/session started before this kernel (including a restart), enter its elapsed age.
SESSION_ALREADY_ELAPSED_HOURS = 0.0
clock_path = Path("/kaggle/temp/researchdesk-session-clock.json")
if "_NOTEBOOK_STARTED_MONOTONIC" not in globals():
    try:
        import psutil
        kernel_age = max(0.0, time.time() - psutil.Process(os.getpid()).create_time())
    except ImportError:
        kernel_age = 0.0
    _NOTEBOOK_STARTED_MONOTONIC = time.monotonic() - max(
        kernel_age, SESSION_ALREADY_ELAPSED_HOURS * 3600
    )
    if clock_path.is_file():
        saved_start = json.loads(clock_path.read_text())["started_epoch"]
        _NOTEBOOK_STARTED_MONOTONIC = min(
            _NOTEBOOK_STARTED_MONOTONIC, time.monotonic() - max(0, time.time() - saved_start)
        )
_NOTEBOOK_STARTED_MONOTONIC = min(
    _NOTEBOOK_STARTED_MONOTONIC, time.monotonic() - SESSION_ALREADY_ELAPSED_HOURS * 3600
)
clock_path.parent.mkdir(parents=True, exist_ok=True)
clock_path.write_text(json.dumps({
    "started_epoch": time.time() - (time.monotonic() - _NOTEBOOK_STARTED_MONOTONIC)
}))
# Run in a fresh Kaggle GPU session, before importing torch/transformers.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["HF_HOME"] = "/kaggle/temp/researchdesk-hf"
# Keep Kaggle's CUDA PyTorch. Verify imports in a clean child process first.
environment_status = install_training_environment(
    "/kaggle/temp/researchdesk-environment.json"
)
_RESEARCHDESK_ENV_READY = environment_status["ready"]

**Ngân sách thời gian:** dừng tối ưu ở giờ 10, dừng đánh giá ở giờ 11, dành giờ cuối để lưu và đóng gói. Timer tính cả cài đặt/tải dữ liệu. Nếu GPU/session đã chạy trước kernel này, đặt SESSION_ALREADY_ELAPSED_HOURS trong cell đầu. Đây là dừng mềm giữa các batch, không bảo đảm trước sự cố hoặc khi Kaggle cưỡng chế ngắt session. Lưu Kaggle Version/Output để giữ artifact.

### 2. Hàm hỗ trợ
Cell này chứa mã đầy đủ, đã nhúng sẵn để notebook chạy độc lập.

In [ ]:
if globals().get("_RESEARCHDESK_ENV_READY", False):
    """QASPER paragraphs and gold evidence, preserving official paper-level splits."""

    import hashlib
    import json
    import unicodedata
    from collections import Counter
    from pathlib import Path
    from typing import Any


    def read_jsonl(path: Path) -> list[dict[str, Any]]:
        return [
            json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()
        ]


    def write_jsonl(path: Path, rows: list[dict[str, Any]]) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(
            "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in rows), encoding="utf-8"
        )


    def file_hash(path: Path) -> str:
        return hashlib.sha256(path.read_bytes()).hexdigest()


    def normalize(text: str) -> str:
        return " ".join(unicodedata.normalize("NFC", text).split())


    def records(value: Any) -> list[dict[str, Any]]:
        """HF's columnar Sequence and official raw list have the same canonical representation."""
        if isinstance(value, list):
            return value
        if not value:
            return []
        lengths = {len(v) for v in value.values()}
        if len(lengths) != 1:
            raise ValueError("Malformed columnar sequence")
        return [{k: v[i] for k, v in value.items()} for i in range(next(iter(lengths)))]


    def prepare_qasper(source: Path, output: Path) -> dict[str, Any]:
        corpus: list[dict[str, Any]] = []
        queries: dict[str, list[dict[str, Any]]] = {}
        pairs: list[dict[str, Any]] = []
        paper_splits: dict[str, str] = {}
        stats: Counter[str] = Counter()
        hashes: dict[str, str] = {}
        for split in ("train", "validation", "test"):
            path = source / f"{split}.json"
            hashes[split] = file_hash(path)
            raw = json.loads(path.read_text(encoding="utf-8"))
            papers = raw.items() if isinstance(raw, dict) else ((r["id"], r) for r in raw)
            queries[split] = []
            for paper_id, paper in papers:
                paper_id = str(paper_id)
                if paper_id in paper_splits:
                    raise ValueError(f"Paper leakage or duplicate: {paper_id}")
                paper_splits[paper_id] = split
                stats[f"{split}_papers"] += 1
                lookup: dict[str, list[str]] = {}
                paragraphs = [("Abstract", paper.get("abstract", ""))]
                for section in records(paper["full_text"]):
                    paragraphs.extend(
                        (section.get("section_name", ""), p) for p in section["paragraphs"]
                    )
                for index, (section_name, text) in enumerate(paragraphs):
                    if not text.strip():
                        continue
                    cid = paper_id + ":" + str(index)
                    row = {
                        "id": cid,
                        "paper_id": paper_id,
                        "text": text.strip(),
                        "section": section_name,
                        "split": split,
                    }
                    corpus.append(row)
                    lookup.setdefault(normalize(text), []).append(cid)
                for qa in records(paper["qas"]):
                    answers = [a["answer"] for a in records(qa["answers"])]
                    answerable = [a for a in answers if not a.get("unanswerable", False)]
                    gold = list(dict.fromkeys(e for a in answerable for e in a.get("evidence", [])))
                    textual = [e for e in gold if not e.startswith("FLOAT SELECTED")]
                    ids = sorted({cid for e in textual for cid in lookup.get(normalize(e), [])})
                    unmatched = [e for e in textual if normalize(e) not in lookup]
                    # Partially mapped gold would inflate recall: exclude such questions, report why.
                    reason = (
                        "unanswerable"
                        if not answerable
                        else "non_text_evidence"
                        if len(textual) != len(gold)
                        else "unmatched_gold"
                        if unmatched
                        else "no_gold"
                        if not ids
                        else ""
                    )
                    question = {
                        "id": str(qa["question_id"]),
                        "question": qa["question"],
                        "paper_id": paper_id,
                        "relevant_ids": ids,
                        "excluded_reason": reason,
                        "split": split,
                    }
                    queries[split].append(question)
                    stats[f"{split}_{reason or 'eligible'}"] += 1
                    if split == "train" and not reason:
                        pairs.extend(
                            {
                                "query_id": question["id"],
                                "question": question["question"],
                                "positive_id": cid,
                                "paper_id": paper_id,
                                "source": "qasper_gold",
                                "reviewed": True,
                            }
                            for cid in ids
                        )
        # Exact duplicate paragraphs across papers can contaminate evaluation too.
        train_texts = {normalize(c["text"]) for c in corpus if c["split"] == "train"}
        heldout_texts = {normalize(c["text"]) for c in corpus if c["split"] != "train"}
        training_text = {c["id"]: normalize(c["text"]) for c in corpus if c["split"] == "train"}
        original_pairs = len(pairs)
        pairs = [p for p in pairs if training_text[p["positive_id"]] not in heldout_texts]
        stats["train_pairs_excluded_duplicate_text"] = original_pairs - len(pairs)
        stats["train_pairs"] = len(pairs)
        output.mkdir(parents=True, exist_ok=True)
        write_jsonl(output / "corpus.jsonl", corpus)
        write_jsonl(output / "train_pairs.jsonl", pairs)
        for split, rows in queries.items():
            write_jsonl(output / f"{split}_queries.jsonl", rows)
        manifest = {
            "format_version": 2,
            "dataset": "allenai/qasper",
            "language": "en",
            "source_hashes": hashes,
            "paper_split": paper_splits,
            "statistics": dict(stats),
            "cross_split_duplicate_texts": len(train_texts & heldout_texts),
            "corpus_sha256": file_hash(output / "corpus.jsonl"),
            "pairs_sha256": file_hash(output / "train_pairs.jsonl"),
        }
        (output / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
        return manifest


    def load_split(data: Path, split: str) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
        corpus = [r for r in read_jsonl(data / "corpus.jsonl") if r["split"] == split]
        queries = [q for q in read_jsonl(data / f"{split}_queries.jsonl") if not q["excluded_reason"]]
        ids = {r["id"] for r in corpus}
        if len(ids) != len(corpus) or not queries or not corpus:
            raise ValueError("Missing eligible queries/corpus or duplicate corpus IDs")
        if any(not q["relevant_ids"] or not set(q["relevant_ids"]) <= ids for q in queries):
            raise ValueError("Gold evidence outside its split")
        return corpus, queries


    def validated_pairs(data: Path, synthetic: Path | None = None) -> list[dict[str, Any]]:
        corpus = {r["id"]: r for r in read_jsonl(data / "corpus.jsonl")}
        heldout_texts = {normalize(c["text"]) for c in corpus.values() if c["split"] != "train"}
        pairs = read_jsonl(data / "train_pairs.jsonl")
        if synthetic:
            pairs += read_jsonl(synthetic)
        result, seen = [], set()
        for pair in pairs:
            if pair.get("reviewed") is not True:
                raise ValueError("Synthetic data must be reviewed by a human before training")
            positive = corpus.get(pair["positive_id"])
            if not positive or positive["split"] != "train" or positive["paper_id"] != pair["paper_id"]:
                raise ValueError("Training evidence leaks held-out papers or has invalid provenance")
            if normalize(positive["text"]) in heldout_texts:
                raise ValueError("Training evidence duplicates held-out text")
            key = (pair["question"], pair["positive_id"])
            if key not in seen:
                seen.add(key)
                result.append({**pair, "positive": positive["text"]})
        if not result:
            raise ValueError("No valid training pairs")
        return result

In [ ]:
if globals().get("_RESEARCHDESK_ENV_READY", False):
    """Standalone helpers embedded into the two Kaggle notebooks at build time.

    Only standard-library imports run on import. Model/data downloads and GPU work are
    explicit notebook cells. The generated notebooks do not import this local file.
    """

    import hashlib
    import importlib.metadata
    import json
    import math
    import random
    import re
    import stat
    import time
    import zipfile
    from collections import Counter, defaultdict
    from pathlib import Path



    class NotebookDeadlineReached(RuntimeError):
        """A cooperative deadline, raised between model batches rather than mid-write."""


    class NotebookTimeBudget:
        def __init__(
            self,
            *,
            started=None,
            session_hours=12,
            train_stop_hours=10,
            export_reserve_hours=1,
            clock=None,
        ):
            self.clock = clock or time.monotonic
            self.started = self.clock() if started is None else started
            if not 0 < train_stop_hours < session_hours - export_reserve_hours < session_hours:
                raise ValueError("Require 0 < train stop < evaluation stop < session limit")
            self.session_hours = session_hours
            self.train_stop_hours = train_stop_hours
            self.export_reserve_hours = export_reserve_hours

        def training_expired(self):
            return self.clock() - self.started >= self.train_stop_hours * 3600

        def evaluation_expired(self):
            return (
                self.clock() - self.started >= (self.session_hours - self.export_reserve_hours) * 3600
            )

        def snapshot(self):
            elapsed = self.clock() - self.started
            return {
                "session_limit_hours": self.session_hours,
                "train_stop_after_hours": self.train_stop_hours,
                "evaluation_stop_after_hours": self.session_hours - self.export_reserve_hours,
                "export_reserve_hours": self.export_reserve_hours,
                "elapsed_hours": elapsed / 3600,
                "training_expired": self.training_expired(),
                "evaluation_expired": self.evaluation_expired(),
                "policy": "cooperative checks between batches; not a hard process timeout",
            }


    def evaluate_with_budget(model, evaluate, budget=None):
        """Abort an incomplete evaluation at a model-batch boundary; never report partial metrics."""
        if budget is None:
            return evaluate()
        if budget.evaluation_expired():
            raise NotebookDeadlineReached("Evaluation budget exhausted; reserve time for export")
        if hasattr(model, "model"):  # CrossEncoder
            target = model.model
        else:  # SentenceTransformer: encode calls forward directly on the outer module.
            first = model[0]
            target = getattr(first, "auto_model", first)

        def check(module, inputs):
            if budget.evaluation_expired():
                raise NotebookDeadlineReached("Evaluation stopped at a batch boundary")

        handle = target.register_forward_pre_hook(check)
        try:
            return evaluate()
        finally:
            handle.remove()


    def make_time_budget_callback(budget, checkpoint_minutes=15):
        from transformers import TrainerCallback

        class TimeBudgetCallback(TrainerCallback):
            def __init__(self):
                self.last_save = time.monotonic()

            def on_step_end(self, args, state, control, **kwargs):
                now = time.monotonic()
                if now - self.last_save >= checkpoint_minutes * 60:
                    control.should_save = True
                    self.last_save = now
                if budget.training_expired():
                    control.should_training_stop = True
                    control.should_save = True
                    control.should_evaluate = False
                return control

            def on_epoch_end(self, args, state, control, **kwargs):
                if budget.training_expired():
                    control.should_training_stop = True
                if budget.evaluation_expired():
                    control.should_evaluate = False
                return control

        return TimeBudgetCallback()


    def download_qasper(destination, revision=""):
        from datasets import load_dataset
        from huggingface_hub import HfApi, hf_hub_download

        destination = Path(destination)
        api = HfApi()
        revision = revision or api.dataset_info("allenai/qasper", revision="refs/convert/parquet").sha
        files = api.list_repo_files("allenai/qasper", repo_type="dataset", revision=revision)
        paths = {
            split: [
                hf_hub_download("allenai/qasper", name, repo_type="dataset", revision=revision)
                for name in files
                if name.endswith(".parquet") and f"/{split}/" in name
            ]
            for split in ("train", "validation", "test")
        }
        if any(not names for names in paths.values()):
            raise ValueError("Official QASPER parquet conversion is missing a split")
        dataset = load_dataset("parquet", data_files=paths)
        destination.mkdir(parents=True, exist_ok=True)
        for split in paths:
            rows = {row["id"]: row for row in dataset[split]}
            (destination / f"{split}.json").write_text(
                json.dumps(rows, ensure_ascii=False), encoding="utf-8"
            )
        metadata = {"dataset": "allenai/qasper", "revision": revision, "license": "CC-BY-4.0"}
        write_json(destination / "download.json", metadata)
        return metadata


    def write_json(path, value):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(value, indent=2, ensure_ascii=False, default=str), encoding="utf-8")


    def package_versions():
        return {
            name: importlib.metadata.version(name)
            for name in (
                "torch",
                "sentence-transformers",
                "transformers",
                "datasets",
                "accelerate",
                "huggingface-hub",
                "numpy",
                "pyarrow",
            )
        }


    def unzip_checkpoint(archive_path, destination):
        destination = Path(destination).resolve()
        destination.mkdir(parents=True, exist_ok=True)
        if any(destination.iterdir()):
            raise ValueError("Choose an empty checkpoint extraction directory")
        with zipfile.ZipFile(archive_path) as archive:
            for info in archive.infolist():
                target = (destination / info.filename).resolve()
                if not target.is_relative_to(destination) or stat.S_ISLNK(info.external_attr >> 16):
                    raise ValueError("Unsafe path or symlink inside checkpoint ZIP")
            archive.extractall(destination)
        if not (destination / "config.json").is_file():
            raise ValueError("ZIP must have config.json at its root, not inside another folder")
        return str(destination)


    def export_checkpoint(model_dir, archive_path, experiment_dir, experiment_zip):
        """ZIP root is the deployable model folder; trainer state is a separate artifact."""
        model_dir, experiment_dir = Path(model_dir), Path(experiment_dir)
        if not (model_dir / "config.json").is_file():
            raise ValueError("Incomplete model: config.json missing")
        weights = list(model_dir.glob("*.safetensors")) + list(model_dir.glob("pytorch_model*.bin"))
        if not weights or not (model_dir / "research_model.json").is_file():
            raise ValueError("Incomplete model: weights or training manifest missing")
        paths = [
            p
            for p in sorted(model_dir.rglob("*"))
            if p.is_file()
            and "checkpoints" not in p.relative_to(model_dir).parts
            and p.name != "artifact_manifest.json"
        ]
        checksum = {
            "format_version": 1,
            "sha256": {p.relative_to(model_dir).as_posix(): file_hash(p) for p in paths},
        }
        write_json(model_dir / "artifact_manifest.json", checksum)
        with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as archive:
            for path in paths:
                archive.write(path, path.relative_to(model_dir).as_posix())
            archive.write(model_dir / "artifact_manifest.json", "artifact_manifest.json")
        with zipfile.ZipFile(experiment_zip, "w", zipfile.ZIP_DEFLATED) as archive:
            for path in sorted(experiment_dir.rglob("*")):
                if path.is_file():
                    archive.write(path, path.relative_to(experiment_dir).as_posix())
            for path in model_dir.glob("*.csv"):
                archive.write(path, "training/" + path.name)
            archive.write(model_dir / "research_model.json", "research_model.json")
        return str(archive_path), str(experiment_zip)


    class NotebookBM25:
        """Inverted-index BM25; does not re-tokenize the corpus for every query."""

        def __init__(self, texts):
            self.lengths, self.postings = [], defaultdict(list)
            for i, text in enumerate(texts):
                counts = Counter(re.findall(r"\w+", text.casefold()))
                self.lengths.append(sum(counts.values()))
                for term, count in counts.items():
                    self.postings[term].append((i, count))
            self.average = max(sum(self.lengths) / max(len(texts), 1), 1)

        def rank(self, query, count):
            scores = defaultdict(float)
            for term in set(re.findall(r"\w+", query.casefold())):
                postings = self.postings.get(term, [])
                idf = math.log(1 + (len(self.lengths) - len(postings) + 0.5) / (len(postings) + 0.5))
                for i, tf in postings:
                    scores[i] += (
                        idf * tf * 2.5 / (tf + 1.5 * (0.25 + 0.75 * self.lengths[i] / self.average))
                    )
            return sorted(scores, key=lambda i: (-scores[i], i))[:count]


    def build_candidates(
        data, split, *, count=40, retriever_path="", batch_size=16, device="cuda", time_budget=None
    ):
        """Use actual retrieval results, NEVER insert gold passages during evaluation."""
        import numpy as np

        corpus, queries = load_split(Path(data), split)
        index = NotebookBM25([c["text"] for c in corpus])
        matrix, query_vectors = None, None
        if retriever_path:
            import torch
            from sentence_transformers import SentenceTransformer

            retriever = SentenceTransformer(retriever_path, device=device, trust_remote_code=False)
            retriever.max_seq_length = 512
            matrix = evaluate_with_budget(
                retriever,
                lambda: retriever.encode(
                    ["passage: " + c["text"] for c in corpus],
                    batch_size=batch_size,
                    normalize_embeddings=True,
                    show_progress_bar=True,
                    convert_to_numpy=True,
                ),
                time_budget,
            )
            query_vectors = evaluate_with_budget(
                retriever,
                lambda: retriever.encode(
                    ["query: " + q["question"] for q in queries],
                    batch_size=batch_size,
                    normalize_embeddings=True,
                    show_progress_bar=True,
                    convert_to_numpy=True,
                ),
                time_budget,
            )
            del retriever
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        samples = []
        for qi, query in enumerate(queries):
            if time_budget and time_budget.evaluation_expired():
                raise NotebookDeadlineReached("Candidate generation stopped; no partial evaluation")
            lexical = index.rank(query["question"], max(100, count))
            if matrix is not None:
                values = matrix @ query_vectors[qi]
                take = min(max(100, count), len(corpus))
                dense = np.argpartition(-values, take - 1)[:take]
                dense = sorted(dense.tolist(), key=lambda i: (-float(values[i]), i))
                fused = defaultdict(float)
                for lane in (lexical, dense):
                    for rank, i in enumerate(lane, 1):
                        fused[i] += 1 / (60 + rank)
                order = sorted(fused, key=lambda i: (-fused[i], i))[:count]
            else:
                order = lexical[:count]
            samples.append(
                {
                    "query_id": query["id"],
                    "query": query["question"],
                    "relevant_ids": query["relevant_ids"],
                    "documents": [{"id": corpus[i]["id"], "text": corpus[i]["text"]} for i in order],
                }
            )
        return samples


    def rank_metrics(ids, gold, k=10):
        ids = list(dict.fromkeys(ids))[:k]
        hits = [rank for rank, cid in enumerate(ids, 1) if cid in gold]
        ideal = sum(1 / math.log2(i + 1) for i in range(1, min(len(gold), k) + 1))
        return {
            f"recall@{k}": len(hits) / len(gold),
            f"mrr@{k}": 1 / hits[0] if hits else 0.0,
            f"ndcg@{k}": sum(1 / math.log2(i + 1) for i in hits) / ideal,
        }


    def evaluate_candidates(model, samples, batch_size=8):
        """Include queries whose gold is completely absent from the candidate list."""
        totals, rows = defaultdict(list), []
        for sample in samples:
            docs = sample["documents"]
            started = time.perf_counter()
            if model is None:
                order = list(range(len(docs)))
            elif docs:
                scores = model.predict(
                    [(sample["query"], d["text"]) for d in docs],
                    batch_size=batch_size,
                    show_progress_bar=False,
                )
                if len(scores) != len(docs) or any(not math.isfinite(float(s)) for s in scores):
                    raise ValueError("Invalid reranker scores")
                order = sorted(range(len(docs)), key=lambda i: (-float(scores[i]), docs[i]["id"]))
            else:
                order = []
            ranked = [docs[i]["id"] for i in order]
            gold = set(sample["relevant_ids"])
            metrics = rank_metrics(ranked, gold)
            metrics["candidate_recall"] = len(gold.intersection(d["id"] for d in docs)) / len(gold)
            for name, value in metrics.items():
                totals[name].append(value)
            rows.append(
                {
                    "query_id": sample["query_id"],
                    "ranked_ids": ranked[:10],
                    "metrics": metrics,
                    "rerank_ms": 1000 * (time.perf_counter() - started),
                }
            )
        if not rows:
            raise ValueError("No evaluation queries")
        return {
            "metrics": {k: sum(v) / len(v) for k, v in totals.items()},
            "queries": len(rows),
            "per_query": rows,
            "gold_injection": False,
            "scope": "all paragraphs in the selected official split",
        }


    def make_reranker_pairs(data, *, negatives_per_query=3, seed=42, reviewed_path="", synthetic=None):
        """Default negatives are cross-paper WEAK labels, never claimed to be human-reviewed.

        Reviewed mode accepts JSONL: query_id, negative_id, label=0, reviewed=true.
        All provenance is checked again; held-out text is never used for training.
        """
        if negatives_per_query < 1:
            raise ValueError("negatives_per_query must be positive")
        data = Path(data)
        all_corpus = read_jsonl(data / "corpus.jsonl")
        heldout = {normalize(c["text"]) for c in all_corpus if c["split"] != "train"}
        corpus = [
            c for c in all_corpus if c["split"] == "train" and normalize(c["text"]) not in heldout
        ]
        by_id = {c["id"]: c for c in corpus}
        grouped = {}
        for p in validated_pairs(data, Path(synthetic) if synthetic else None):
            qid = (
                p.get("query_id") or "custom_" + hashlib.sha256(p["question"].encode()).hexdigest()[:16]
            )
            key = (qid, p["question"], p["paper_id"])
            grouped.setdefault(key, set()).add(p["positive_id"])
        reviewed = defaultdict(list)
        valid_qids = {key[0] for key in grouped}
        if reviewed_path:
            for row in read_jsonl(Path(reviewed_path)):
                if row.get("reviewed") is not True or row.get("label") != 0:
                    raise ValueError("Reviewed negative requires reviewed=true and label=0")
                if row.get("query_id") not in valid_qids or row.get("negative_id") not in by_id:
                    raise ValueError(
                        "Reviewed negative has unknown query or held-out/duplicate passage"
                    )
                reviewed[row["query_id"]].append(row["negative_id"])
        lexical = NotebookBM25([c["text"] for c in corpus])
        rng = random.Random(seed)
        rows, review_rows = [], []
        for (qid, question, paper_id), gold in sorted(grouped.items()):
            gold_texts = {normalize(by_id[cid]["text"]) for cid in gold}
            if reviewed_path:
                chosen = list(dict.fromkeys(reviewed[qid]))[:negatives_per_query]
                if any(cid in gold or normalize(by_id[cid]["text"]) in gold_texts for cid in chosen):
                    raise ValueError("A reviewed negative is actually a known positive")
            else:
                ranked = lexical.rank(question, 200)
                selected = set(ranked)
                remainder = [i for i in range(len(corpus)) if i not in selected]
                rng.shuffle(remainder)
                chosen = []
                for i in [*ranked, *remainder]:
                    c = corpus[i]
                    if c["paper_id"] != paper_id and normalize(c["text"]) not in gold_texts:
                        chosen.append(c["id"])
                        if len(chosen) == negatives_per_query:
                            break
            if not chosen:
                continue
            for cid in sorted(gold):
                rows.append(
                    {
                        "query_id": qid,
                        "query": question,
                        "passage_id": cid,
                        "passage": by_id[cid]["text"],
                        "label": 1.0,
                        "source": "reviewed_positive",
                    }
                )
            for cid in chosen:
                rows.append(
                    {
                        "query_id": qid,
                        "query": question,
                        "passage_id": cid,
                        "passage": by_id[cid]["text"],
                        "label": 0.0,
                        "source": "human_reviewed" if reviewed_path else "cross_paper_weak",
                    }
                )
                review_rows.append(
                    {
                        "query_id": qid,
                        "question": question,
                        "negative_id": cid,
                        "passage": by_id[cid]["text"],
                        "label": 0,
                        "reviewed": bool(reviewed_path),
                    }
                )
        if not rows or not any(r["label"] == 0 for r in rows):
            raise ValueError("No usable negatives; supply reviewed negatives from training papers")
        return rows, review_rows


    def train_reranker(
        base,
        output,
        rows,
        validation_samples,
        *,
        revision="",
        epochs=2,
        batch_size=8,
        accumulation=2,
        learning_rate=2e-5,
        max_length=512,
        seed=42,
        device="cuda",
        time_budget=None,
        checkpoint_minutes=15,
    ):
        """Real PyTorch BCE optimization; choose checkpoint by validation nDCG@10."""
        import numpy as np
        import torch
        from sentence_transformers import CrossEncoder
        from torch.utils.data import DataLoader
        from transformers import get_linear_schedule_with_warmup, set_seed

        if epochs < 1 or batch_size < 1 or accumulation < 1:
            raise ValueError("epochs, batch_size and accumulation must be positive integers")
        if device == "cuda" and not torch.cuda.is_available():
            raise ValueError("CUDA unavailable; enable Kaggle GPU")
        output = Path(output)
        if output.exists() and any(output.iterdir()):
            raise ValueError("Output must be empty; use a new RUN_NAME")
        output.mkdir(parents=True, exist_ok=True)
        set_seed(seed)
        options = {"device": device, "max_length": max_length, "trust_remote_code": False}
        if revision:
            options["revision"] = revision
        model = CrossEncoder(base, **options)
        if model.model.config.num_labels != 1:
            raise ValueError("Reranker must have one relevance logit")
        original_count = len(rows)
        # Positive evidence truncated at the answer tail must not silently retain label=1.
        rows = [
            r
            for r in rows
            if r["label"] == 0
            or len(model.tokenizer(r["query"], r["passage"], truncation=False)["input_ids"])
            <= max_length
        ]
        positives = sum(r["label"] == 1 for r in rows)
        negatives = len(rows) - positives
        if not positives or not negatives:
            raise ValueError("Need both non-truncated positives and negatives")

        def collate(batch):
            features = model.tokenizer(
                [r["query"] for r in batch],
                [r["passage"] for r in batch],
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            )
            return features, torch.tensor([r["label"] for r in batch], dtype=torch.float32)

        loader = DataLoader(
            rows,
            batch_size=batch_size,
            shuffle=True,
            collate_fn=collate,
            num_workers=0,
            generator=torch.Generator().manual_seed(seed),
        )
        before = None
        try:
            before = evaluate_with_budget(
                model, lambda: evaluate_candidates(model, validation_samples, batch_size), time_budget
            )
        except NotebookDeadlineReached:
            print("Baseline validation skipped/incomplete due to the session deadline.", flush=True)
        write_json(output / "validation_before.json", before)
        optimizer = torch.optim.AdamW(model.model.parameters(), lr=learning_rate, weight_decay=0.01)
        updates = math.ceil(len(loader) / accumulation) * epochs
        scheduler = get_linear_schedule_with_warmup(optimizer, int(updates * 0.1), updates)
        use_amp = device == "cuda"
        scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
        loss_fn = torch.nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor(negatives / positives, dtype=torch.float32, device=device)
        )
        best, best_epoch, history = -math.inf, 0, []
        optimizer_steps, stopped = 0, False
        last_save = time.monotonic()
        recovery = output.parent / "recovery-reranker"
        for epoch in range(epochs):
            model.model.train()
            optimizer.zero_grad(set_to_none=True)
            losses = []
            for step, (features, labels) in enumerate(loader):
                if time_budget and time_budget.training_expired():
                    stopped = True
                    break
                features = {key: value.to(device) for key, value in features.items()}
                labels = labels.to(device)
                # Scale the final partial accumulation group correctly.
                group_size = min(accumulation, len(loader) - (step // accumulation) * accumulation)
                with torch.autocast(device_type="cuda" if use_amp else "cpu", enabled=use_amp):
                    logits = model.model(**features).logits.reshape(-1)
                    raw_loss = loss_fn(logits.float(), labels)
                    loss = raw_loss / group_size
                if not torch.isfinite(raw_loss):
                    raise ValueError("Non-finite training loss; reduce learning rate or disable AMP")
                scaler.scale(loss).backward()
                losses.append(float(raw_loss.detach().cpu()))
                if (step + 1) % accumulation == 0 or step + 1 == len(loader):
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.model.parameters(), 1.0)
                    scale_before = scaler.get_scale()
                    scaler.step(optimizer)
                    scaler.update()
                    if scaler.get_scale() >= scale_before:
                        scheduler.step()
                        optimizer_steps += 1
                    optimizer.zero_grad(set_to_none=True)
                    if time.monotonic() - last_save >= checkpoint_minutes * 60:
                        model.save_pretrained(str(recovery), safe_serialization=True)
                        write_json(
                            recovery / "checkpoint_status.json",
                            {
                                "selection": "last_unvalidated",
                                "optimizer_steps": optimizer_steps,
                                "epoch": epoch + (step + 1) / len(loader),
                                "resume_optimizer_supported": False,
                            },
                        )
                        last_save = time.monotonic()
            optimizer.zero_grad(set_to_none=True)
            model.model.eval()
            validation = None
            try:
                validation = evaluate_with_budget(
                    model,
                    lambda: evaluate_candidates(model, validation_samples, batch_size),
                    time_budget,
                )
            except NotebookDeadlineReached:
                print("Validation stopped/incomplete; no partial metric is reported.", flush=True)
            row = {
                "epoch": epoch + 1,
                "train_loss": float(np.mean(losses)) if losses else None,
                "validation": validation["metrics"] if validation else None,
                "partial_epoch": stopped,
            }
            history.append(row)
            print(json.dumps(row), flush=True)
            if validation and optimizer_steps and validation["metrics"]["ndcg@10"] > best:
                best, best_epoch = validation["metrics"]["ndcg@10"], epoch + 1
                model.save_pretrained(str(output), safe_serialization=True)
                write_json(output / "validation_after.json", validation)
            # Kaggle Saved Version / Output can preserve this between sessions.
            # Notebook deliberately does not silently resume a different experiment.
            write_json(output / "training_history.json", history)
            if stopped or (time_budget and time_budget.training_expired()):
                stopped = True
                break
        selection = "best_validation_checkpoint" if best_epoch else "last_unvalidated"
        if not best_epoch:
            model.save_pretrained(str(output), safe_serialization=True)
            write_json(output / "validation_after.json", None)
        if not optimizer_steps:
            selection = "base_not_trained"
        manifest = {
            "format_version": 2,
            "model_family": "cross_encoder",
            "fine_tuned": optimizer_steps > 0,
            "base_model": base,
            "revision": revision,
            "best_epoch": best_epoch,
            "loss": "BCEWithLogitsLoss",
            "max_seq_length": max_length,
            "rows_used": len(rows),
            "positive_rows": positives,
            "negative_rows": negatives,
            "rows_excluded_truncation": original_count - len(rows),
            "negative_sources": dict(Counter(r["source"] for r in rows if r["label"] == 0)),
            "config": {
                "epochs": epochs,
                "batch_size": batch_size,
                "accumulation": accumulation,
                "learning_rate": learning_rate,
                "seed": seed,
                "device": device,
            },
            "validation_before": before["metrics"] if before else None,
            "validation_after": json.loads((output / "validation_after.json").read_text()),
            "optimizer_steps": optimizer_steps,
            "selection": selection,
            "stop_reason": "time_budget" if stopped else "epochs_completed",
            "time_budget": time_budget.snapshot() if time_budget else None,
            "history": history,
            "versions": package_versions(),
        }
        if manifest["validation_after"]:
            manifest["validation_after"] = manifest["validation_after"]["metrics"]
        write_json(output / "research_model.json", manifest)
        return manifest

In [ ]:
if globals().get("_RESEARCHDESK_ENV_READY", False):
    import gc, json, shutil
    from datetime import datetime, timezone
    from pathlib import Path
    import numpy as np
    import torch
    from huggingface_hub import HfApi
    from sentence_transformers import SentenceTransformer, CrossEncoder
    assert torch.cuda.is_available(), "Enable a GPU in Kaggle Settings before training."
    print("Python:", sys.version)
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
    print("Packages:", json.dumps(package_versions(), indent=2))
    print("One GPU is used. Two T4 cards do not form one shared VRAM pool.")

### 3. Cấu hình
Nếu OOM, giảm `MINI_BATCH_SIZE` còn 2 rồi 1 và dùng RUN_NAME mới; có thể chọn E5-small nếu cần giảm tải. So sánh trước–sau phải cùng model nền. Logical batch khác gradient accumulation; CachedMNRL giữ tập negatives rộng hơn.

In [ ]:
if globals().get("_RESEARCHDESK_ENV_READY", False):
    # Edit only these settings for a first experiment.
    BASE_MODEL = "intfloat/multilingual-e5-base"  # same size as the current app baseline
    BASE_REVISION = ""  # blank resolves and records a commit SHA
    RUN_NAME = "e5-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
    EPOCHS = 2
    BATCH_SIZE = 32             # logical contrastive batch, not all encoded at once
    MINI_BATCH_SIZE = 4         # CachedMNRL encodes small groups to reduce VRAM
    MAX_SEQ_LENGTH = 512
    LEARNING_RATE = 2e-5
    SEED = 42
    QASPER_REVISION = ""
    PREPARED_DATA = ""         # optional prepared dataset directory under /kaggle/input
    SYNTHETIC_PATH = ""        # optional HUMAN-REVIEWED training JSONL
    RUN_FINAL_TEST = False     # set True only after freezing all hyperparameters
    RUN = Path("/kaggle/working") / RUN_NAME
    assert not RUN.exists(), "Choose a new RUN_NAME; do not overwrite an experiment."
    (RUN / "experiment").mkdir(parents=True)
    MODEL_OUT = RUN / "model"

    # All limits are measured from the first setup cell (or the older kernel/session age).
    SESSION_LIMIT_HOURS = 12
    TRAIN_STOP_AFTER_HOURS = 10
    EXPORT_RESERVE_HOURS = 1  # evaluation stops at hour 11; preserve the last hour for export
    CHECKPOINT_EVERY_MINUTES = 15
    TIME_BUDGET = NotebookTimeBudget(
        started=_NOTEBOOK_STARTED_MONOTONIC, session_hours=SESSION_LIMIT_HOURS,
        train_stop_hours=TRAIN_STOP_AFTER_HOURS, export_reserve_hours=EXPORT_RESERVE_HOURS,
    )
    write_json(RUN / "experiment" / "time_budget.json", TIME_BUDGET.snapshot())
    print("Session time budget:", json.dumps(TIME_BUDGET.snapshot(), indent=2))

### Dữ liệu và chia tập
Mặc định tải **QASPER chính thức, tiếng Anh**, pin commit SHA của bản Parquet;
không gọi LLM API. Giữ nguyên split theo paper, loại gold hình/bảng, gold không
khớp đầy đủ và positive train trùng text held-out. Validation chọn checkpoint;
test chỉ chạy một lần sau khi chốt cấu hình. Không kết luận chất lượng tiếng Việt từ QASPER.

Để dùng corpus riêng, đặt `PREPARED_DATA` tới thư mục Kaggle Input chứa
`corpus.jsonl`, `train_pairs.jsonl`, `{train,validation,test}_queries.jsonl`, `manifest.json`
theo định dạng Research Desk. Split tài liệu trước khi tạo câu hỏi. Có thể thêm
JSONL train tiếng Việt bằng `SYNTHETIC_PATH`: mỗi dòng có `question`, `positive_id`,
`paper_id`, `reviewed=true`; đoạn positive phải thuộc corpus train, được người đọc kiểm duyệt.

In [ ]:
if globals().get("_RESEARCHDESK_ENV_READY", False):
    DATA = Path(PREPARED_DATA) if PREPARED_DATA else Path("/kaggle/temp/researchdesk-qasper-prepared")
    if not PREPARED_DATA:
        raw = Path("/kaggle/temp/researchdesk-qasper-raw")
        download_info = download_qasper(raw, QASPER_REVISION)
        dataset_manifest = prepare_qasper(raw, DATA)
        write_json(RUN / "experiment" / "download.json", download_info)
    else:
        required = ["corpus.jsonl", "train_pairs.jsonl", "manifest.json"]
        required += [f"{s}_queries.jsonl" for s in ("train", "validation", "test")]
        assert all((DATA / name).is_file() for name in required), "Incomplete prepared dataset"
        dataset_manifest = json.loads((DATA / "manifest.json").read_text(encoding="utf-8"))
        # Revalidate IDs, split provenance and exact duplicate evidence before any training.
        validated_pairs(DATA, Path(SYNTHETIC_PATH) if SYNTHETIC_PATH else None)
    for split in ("train", "validation", "test"):
        corpus, questions = load_split(DATA, split)
        print(split, "paragraphs:", len(corpus), "eligible questions:", len(questions))
    print("Dataset statistics:", dataset_manifest.get("statistics", {}))
    write_json(RUN / "experiment" / "dataset_manifest.json", dataset_manifest)
    write_json(RUN / "experiment" / "packages.json", package_versions())

### 4. Fine-tune và validation
Không reset giới hạn 512 token. Loại positive bị truncation, dùng NO_DUPLICATES sampler; có thể còn false negatives giữa các đoạn tương đương. Chọn epoch theo metric validation, ghi hash và seed. Giữ toàn bộ thư mục run nếu muốn lưu cả checkpoint trung gian.

In [ ]:
if globals().get("_RESEARCHDESK_ENV_READY", False):
    """Notebook E5 training: cached negatives, cooperative deadlines and explicit selection status."""

    import importlib.metadata
    import json
    from dataclasses import asdict, dataclass
    from pathlib import Path



    @dataclass
    class TrainConfig:
        data: str
        output: str
        model: str = "intfloat/multilingual-e5-base"
        revision: str = ""
        epochs: float = 2
        batch_size: int = 32
        mini_batch_size: int = 4
        learning_rate: float = 2e-5
        max_seq_length: int = 512
        device: str = "cuda"
        seed: int = 42
        synthetic: str | None = None


    def train(config, time_budget=None, checkpoint_minutes=15):
        import sentence_transformers.losses as losses
        import torch
        from datasets import Dataset
        from sentence_transformers import (
            SentenceTransformer,
            SentenceTransformerTrainer,
            SentenceTransformerTrainingArguments,
        )
        from sentence_transformers.evaluation import InformationRetrievalEvaluator, SentenceEvaluator
        from sentence_transformers.training_args import BatchSamplers
        from transformers import set_seed

        if config.batch_size < 2 or config.mini_batch_size < 1 or config.epochs <= 0:
            raise ValueError("Require batch >= 2, mini batch >= 1 and epochs > 0")
        if config.device == "cuda" and not torch.cuda.is_available():
            raise ValueError("CUDA unavailable; enable Kaggle GPU")
        set_seed(config.seed)
        options = {"device": config.device, "trust_remote_code": False}
        revision = config.revision
        if not Path(config.model).is_dir() and not revision:
            from huggingface_hub import HfApi

            revision = HfApi().model_info(config.model).sha or ""
        if revision:
            options["revision"] = revision
        model = SentenceTransformer(config.model, **options)
        model.max_seq_length = config.max_seq_length
        data = Path(config.data)
        pairs = validated_pairs(data, Path(config.synthetic) if config.synthetic else None)
        original = len(pairs)
        pairs = [
            p
            for p in pairs
            if len(model.tokenizer.encode("query: " + p["question"])) <= config.max_seq_length
            and len(model.tokenizer.encode("passage: " + p["positive"])) <= config.max_seq_length
        ]
        if len(pairs) < 2:
            raise ValueError("Too few non-truncated training pairs")
        dataset = Dataset.from_dict(
            {
                "anchor": ["query: " + p["question"] for p in pairs],
                "positive": ["passage: " + p["positive"] for p in pairs],
            }
        )
        corpus, queries = load_split(data, "validation")
        evaluator = InformationRetrievalEvaluator(
            queries={q["id"]: "query: " + q["question"] for q in queries},
            corpus={c["id"]: "passage: " + c["text"] for c in corpus},
            relevant_docs={q["id"]: set(q["relevant_ids"]) for q in queries},
            name="validation",
            show_progress_bar=False,
            batch_size=config.mini_batch_size,
            write_csv=True,
        )

        class TimedEvaluator(SentenceEvaluator):
            primary_metric = evaluator.primary_metric

            def __call__(self, current_model, *args, **kwargs):
                result = evaluate_with_budget(
                    current_model, lambda: evaluator(current_model, *args, **kwargs), time_budget
                )
                self.primary_metric = evaluator.primary_metric
                return result

        timed = TimedEvaluator()
        output = Path(config.output)
        if output.exists() and any(output.iterdir()):
            raise ValueError("Output must be empty; choose a new RUN_NAME")
        output.mkdir(parents=True, exist_ok=True)
        baseline, after, metrics, steps = None, None, {}, 0
        selection, stop_reason = "base_not_trained", "epochs_completed"
        trainer = None
        try:
            if time_budget and time_budget.training_expired():
                raise NotebookDeadlineReached("Training deadline already reached")
            baseline = timed(model, output_path=str(output))
            if time_budget and time_budget.training_expired():
                raise NotebookDeadlineReached("Training deadline reached during setup/validation")
            args = SentenceTransformerTrainingArguments(
                output_dir=str(output / "checkpoints"),
                num_train_epochs=config.epochs,
                per_device_train_batch_size=config.batch_size,
                per_device_eval_batch_size=config.mini_batch_size,
                learning_rate=config.learning_rate,
                warmup_ratio=0.1,
                fp16=config.device == "cuda",
                bf16=False,
                batch_sampler=BatchSamplers.NO_DUPLICATES,
                eval_strategy="epoch",
                save_strategy="epoch",
                save_total_limit=2,
                load_best_model_at_end=True,
                metric_for_best_model=timed.primary_metric,
                greater_is_better=True,
                logging_steps=10,
                seed=config.seed,
                report_to="none",
                dataloader_num_workers=0,
            )
            trainer = SentenceTransformerTrainer(
                model=model,
                args=args,
                train_dataset=dataset,
                loss=losses.CachedMultipleNegativesRankingLoss(
                    model, mini_batch_size=config.mini_batch_size
                ),
                evaluator=timed,
                callbacks=[make_time_budget_callback(time_budget, checkpoint_minutes)]
                if time_budget
                else [],
            )
            trained = trainer.train()
            metrics = trained.metrics
        except NotebookDeadlineReached as exc:
            stop_reason = "time_budget"
            print(str(exc), flush=True)
            if trainer is not None:
                # Preserve current weights when evaluation was interrupted.
                # This is explicit recovery data, not an evaluated best model.
                trainer.save_model(str(output.parent / "recovery-e5"))
                trainer.save_state()
        if trainer is not None:
            steps = int(trainer.state.global_step)
            checkpoint = trainer.state.best_model_checkpoint
            if checkpoint:
                # Also works when a deadline interrupted Trainer before its normal best-model reload.
                model = SentenceTransformer(checkpoint, **{"device": config.device})
                selection = "best_validation_checkpoint"
            else:
                model = trainer.model
                selection = "last_unvalidated" if steps else "base_not_trained"
            if time_budget and time_budget.training_expired():
                stop_reason = "time_budget"
        try:
            if steps:
                after = timed(model, output_path=str(output))
        except NotebookDeadlineReached:
            print("Final validation skipped/incomplete; no partial metric is reported.", flush=True)
        if after is not None:
            selection = (
                "best_validation_checkpoint" if selection.startswith("best_") else "last_validated"
            )
        model.save_pretrained(str(output))
        manifest = {
            "format_version": 2,
            "model_family": "e5",
            "fine_tuned": steps > 0,
            "loss": "CachedMultipleNegativesRankingLoss",
            "query_prefix": "query: ",
            "passage_prefix": "passage: ",
            "base_model": config.model,
            "revision": revision,
            "max_seq_length": config.max_seq_length,
            "config": asdict(config),
            "pairs_used": len(pairs),
            "pairs_excluded_truncation": original - len(pairs),
            "corpus_sha256": file_hash(data / "corpus.jsonl"),
            "pairs_sha256": file_hash(data / "train_pairs.jsonl"),
            "synthetic_sha256": file_hash(Path(config.synthetic)) if config.synthetic else None,
            "validation_before": baseline,
            "validation_after": after,
            "training_metrics": metrics,
            "optimizer_steps": steps,
            "selection": selection,
            "stop_reason": stop_reason,
            "selected_validation_metric": trainer.state.best_metric if trainer else None,
            "time_budget": time_budget.snapshot() if time_budget else None,
            "versions": {
                name: importlib.metadata.version(name)
                for name in ("torch", "sentence-transformers", "datasets", "transformers", "accelerate")
            },
        }
        (output / "research_model.json").write_text(
            json.dumps(manifest, indent=2, default=str), encoding="utf-8"
        )
        return manifest

In [ ]:
if globals().get("_RESEARCHDESK_ENV_READY", False):
    revision = BASE_REVISION or HfApi().model_info(BASE_MODEL).sha
    config = TrainConfig(
        data=str(DATA), output=str(MODEL_OUT), model=BASE_MODEL, revision=revision,
        epochs=EPOCHS, batch_size=BATCH_SIZE, mini_batch_size=MINI_BATCH_SIZE,
        learning_rate=LEARNING_RATE, max_seq_length=MAX_SEQ_LENGTH,
        device="cuda", seed=SEED, synthetic=SYNTHETIC_PATH or None,
    )
    write_json(RUN / "experiment" / "run_config.json", asdict(config))
    manifest = train(config, time_budget=TIME_BUDGET, checkpoint_minutes=CHECKPOINT_EVERY_MINUTES)
    print("Training status:", {key: manifest[key] for key in (
        "fine_tuned", "optimizer_steps", "selection", "stop_reason"
    )})
    write_json(RUN / "experiment" / "validation.json", {
        "before": manifest["validation_before"], "after": manifest["validation_after"],
    })
    print(json.dumps({key: manifest[key] for key in (
        "pairs_used", "pairs_excluded_truncation", "validation_before", "validation_after"
    )}, indent=2))
    # These are retrieval scores, not accuracy of generated answers. A fine-tune can regress.
    gc.collect()
    torch.cuda.empty_cache()

### 5. Test cuối — mặc định tắt
Chỉ bật `RUN_FINAL_TEST` sau khi chốt cấu hình. So sánh E5 nền và fine-tuned trên cùng toàn bộ corpus test; không chọn model bằng test.

In [ ]:
if globals().get("_RESEARCHDESK_ENV_READY", False):
    if RUN_FINAL_TEST and not TIME_BUDGET.training_expired():
        from sentence_transformers.evaluation import InformationRetrievalEvaluator
        corpus, queries = load_split(DATA, "test")
        evaluator = InformationRetrievalEvaluator(
            queries={q["id"]: "query: " + q["question"] for q in queries},
            corpus={c["id"]: "passage: " + c["text"] for c in corpus},
            relevant_docs={q["id"]: set(q["relevant_ids"]) for q in queries},
            name="test", batch_size=16, show_progress_bar=True,
        )
        test_report = {"split": "test", "corpus_sha256": file_hash(DATA / "corpus.jsonl"),
                       "queries_sha256": file_hash(DATA / "test_queries.jsonl"), "models": {}}
        for label, name in (("base", BASE_MODEL), ("fine_tuned", str(MODEL_OUT))):
            kwargs = {"revision": revision} if label == "base" else {}
            test_model = SentenceTransformer(name, device="cuda", trust_remote_code=False, **kwargs)
            test_model.max_seq_length = MAX_SEQ_LENGTH
            try:
                test_report["models"][label] = evaluate_with_budget(
                    test_model, lambda: evaluator(test_model), TIME_BUDGET
                )
            except NotebookDeadlineReached:
                test_report["incomplete_due_to_time_budget"] = True
                break
            del test_model
            gc.collect()
            torch.cuda.empty_cache()
        write_json(RUN / "experiment" / "test.json", test_report)
        print(json.dumps(test_report, indent=2))
    else:
        print("Test skipped: disabled or insufficient time. Freeze settings before final test.")

### 6. Nạp lại offline trên CPU và xuất model
Kiểm tra này xác nhận checkpoint nạp được; chất lượng phải đọc từ metrics và đánh giá câu trả lời cuối.

In [ ]:
if globals().get("_RESEARCHDESK_ENV_READY", False):
    # Offline CPU reload: this checks packaging/load compatibility, not research quality.
    reload_model = SentenceTransformer(str(MODEL_OUT), device="cpu", local_files_only=True)
    vectors = reload_model.encode(
        ["query: How is evidence retrieved?", "passage: Evidence is retrieved from relevant passages."],
        normalize_embeddings=True, convert_to_numpy=True,
    )
    assert vectors.shape[0] == 2 and np.isfinite(vectors).all()
    assert np.allclose(np.linalg.norm(vectors, axis=1), 1, atol=1e-4)
    write_json(RUN / "experiment" / "reload_check.json", {
        "passed": True, "device": "cpu", "embedding_dimension": int(vectors.shape[1]),
    })
    del reload_model
    gc.collect()

In [ ]:
if globals().get("_RESEARCHDESK_ENV_READY", False):
    checkpoint_zip = Path("/kaggle/working/research-e5-checkpoint.zip")
    experiment_zip = Path("/kaggle/working/research-e5-experiment.zip")
    paths = export_checkpoint(MODEL_OUT, checkpoint_zip, RUN / "experiment", experiment_zip)
    from IPython.display import FileLink, display
    for path in paths:
        print(path)
        display(FileLink(path))
    print("Download both ZIPs from Kaggle Output after saving the notebook version.")

### 7. Đưa vào hệ thống
Tải `research-e5-checkpoint.zip` và `research-e5-experiment.zip`.
Giải nén checkpoint vào `models/multilingual-e5-research`; `modules.json`,
`config.json`, tokenizer, weights và `research_model.json` phải nằm đúng vị trí.

```dotenv
RA_EMBEDDING_PROVIDER=sentence-transformers
RA_EMBEDDING_MODEL=models/multilingual-e5-research
RA_EMBEDDING_FAMILY=e5
RA_EMBEDDING_MAX_SEQ_LENGTH=512
RA_LOCAL_DEVICE=cpu
```

Có thể dùng `cuda` nếu máy đủ VRAM. Restart backend và **Reindex toàn bộ embedding**.
Không trộn vector của model cũ/mới. Notebook 02 có thể dùng ZIP checkpoint này làm Input.
ZIP experiment là bằng chứng thực nghiệm, không phải model để nạp vào app.

Một `.pt` riêng không chứa đủ tokenizer, pooling và cấu hình. Dùng nguyên checkpoint
`save_pretrained` với `model.safetensors`; app hiện tại hỗ trợ trực tiếp định dạng này.
Nguồn: [E5](https://huggingface.co/intfloat/multilingual-e5-base),
[QASPER](https://aclanthology.org/2021.naacl-main.365/),
[CachedMNRL](https://sbert.net/docs/package_reference/sentence_transformer/losses.html).